## **RIPPER ( Repeated Incremental Pruning to Produce Error Reduction )**

**RIPPER** ( Cohen, 1995 ) is the best-known **rule-list learner** after CN2. It works directly on **numeric attributes** ( conditions such as `mean radius > 14.9` ) and learns, for the minority class, rules that are

1. **grown** greedily with the **FOIL information gain**,
2. **pruned** on a held-out part of the data ( *incremental reduced-error pruning* ),
3. accepted only while the **minimum description length ( MDL )** of the rule set does not grow too much,

and ends with a default rule for the majority class. This notebook implements the **IREP\* core with MDL stopping and the final deletion of rules that increase the description length**; the *optimisation* passes of the full algorithm ( replacement and revision of rules ) are **not** included.

**Notebooks of this series:** 1. CN2 Rule Induction · 2. Expert Systems · 3. Case-Based Reasoning · 4. Apriori for Classification ( CBA ) · 5. Simple Rule Learners ( ZeroR, OneR, PRISM ) · 6. RIPPER · 7. Rules from Decision Trees · 8. RuleFit · 9. Fuzzy Rule-Based Classifier

### **0. Environment Setup**

In [1]:
#Import Libs
import sklearn
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from cycler import cycler

In [2]:
#Define Colors Palette and Standards for Illustrations
RED, DARK, GRAY, LIGHT, SOFT = "#CD2027", "#404040", "#7F7F7F", "#D9D9D9", "#E9A0A3"

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

plt.rcParams.update({
    "figure.figsize": (7.5, 4.2),
    "figure.dpi": 100,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.titleweight": "bold",
    "axes.prop_cycle": cycler(color=[RED, DARK, SOFT, GRAY]),
})
pd.set_option("display.precision", 3)
pd.set_option("display.width", 120)

In [3]:
#Helper Functions: Evaluation Reports ( Confusion Matrix, Metrics, ROC and Precision-Recall )
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, log_loss, matthews_corrcoef,
                             confusion_matrix, roc_curve, precision_recall_curve)

def plot_confusion(cm, labels, ax, title="Confusion matrix"):
    ax.imshow(cm, cmap="Reds"); ax.grid(False)
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=45 if len(labels) > 4 else 0, ha="right" if len(labels) > 4 else "center")
    ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=10 if len(labels) > 4 else 13,
                    color="white" if cm[i, j] > cm.max() / 2 else DARK)
    ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(title)

def report_binary(y_true, y_pred, y_score, name="model", labels=("negative", "positive"), plot=True):
    """Confusion matrix, ROC and Precision-Recall curves + a Series with the main metrics (positive class = 1)."""
    y_true, y_pred, y_score = np.asarray(y_true), np.asarray(y_pred), np.asarray(y_score)
    metrics = {"accuracy": accuracy_score(y_true, y_pred), "balanced acc.": balanced_accuracy_score(y_true, y_pred),
               "precision": precision_score(y_true, y_pred, zero_division=0), "recall": recall_score(y_true, y_pred),
               "F1": f1_score(y_true, y_pred), "MCC": matthews_corrcoef(y_true, y_pred),
               "ROC AUC": roc_auc_score(y_true, y_score), "PR AUC": average_precision_score(y_true, y_score)}
    if plot:
        fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
        plot_confusion(confusion_matrix(y_true, y_pred), list(labels), axes[0], f"Confusion matrix – {name}")
        fpr, tpr, _ = roc_curve(y_true, y_score)
        axes[1].plot(fpr, tpr, color=RED, lw=2.4, label=f"AUC = {metrics['ROC AUC']:.3f}"); axes[1].plot([0, 1], [0, 1], color=GRAY, ls="--")
        axes[1].set_xlabel("false positive rate"); axes[1].set_ylabel("true positive rate"); axes[1].set_title("ROC curve"); axes[1].legend(loc="lower right")
        p, r, _ = precision_recall_curve(y_true, y_score)
        axes[2].plot(r, p, color=DARK, lw=2.4, label=f"AP = {metrics['PR AUC']:.3f}"); axes[2].axhline(y_true.mean(), color=GRAY, ls="--")
        axes[2].set_xlabel("recall"); axes[2].set_ylabel("precision"); axes[2].set_title("Precision–Recall curve"); axes[2].legend(loc="lower left")
        plt.tight_layout(); plt.show()
    return pd.Series(metrics, name=name)

def report_multiclass(y_true, y_pred, y_proba, labels, name="model", plot=True):
    """Confusion matrix, one-vs-rest ROC curves and per-class F1 + a Series with the main metrics."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    k = len(labels)
    metrics = {"accuracy": accuracy_score(y_true, y_pred), "balanced acc.": balanced_accuracy_score(y_true, y_pred),
               "macro precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
               "macro recall": recall_score(y_true, y_pred, average="macro"), "macro F1": f1_score(y_true, y_pred, average="macro"),
               "MCC": matthews_corrcoef(y_true, y_pred), "ROC AUC (ovr)": roc_auc_score(y_true, y_proba, multi_class="ovr"),
               "log loss": log_loss(y_true, y_proba, labels=list(range(k)))}
    if plot:
        fig, axes = plt.subplots(1, 3, figsize=(15, 4.4), gridspec_kw={"width_ratios": [1.1, 1, 1]})
        plot_confusion(confusion_matrix(y_true, y_pred, labels=list(range(k))), list(labels), axes[0], f"Confusion matrix – {name}")
        cmap = plt.get_cmap("tab10")
        for c in range(k):
            fpr, tpr, _ = roc_curve((y_true == c).astype(int), y_proba[:, c])
            axes[1].plot(fpr, tpr, color=cmap(c), lw=1.6, label=f"{labels[c]}")
        axes[1].plot([0, 1], [0, 1], color=GRAY, ls="--"); axes[1].set_xlabel("false positive rate"); axes[1].set_ylabel("true positive rate")
        axes[1].set_title(f"ROC one-vs-rest (macro AUC = {metrics['ROC AUC (ovr)']:.3f})"); axes[1].legend(fontsize=7, ncol=2 if k > 5 else 1, loc="lower right")
        f1s = f1_score(y_true, y_pred, average=None)
        axes[2].bar(range(k), f1s, color=RED, width=0.65); axes[2].set_xticks(range(k)); axes[2].set_xticklabels(labels, rotation=45 if k > 4 else 0, ha="right" if k > 4 else "center")
        axes[2].set_ylim(0, 1.05); axes[2].set_ylabel("F1"); axes[2].set_title("F1 per class")
        plt.tight_layout(); plt.show()
    return pd.Series(metrics, name=name)
def annotate_heatmap(ax, mat, fmt="{:.2f}"):
    """Write the values on an imshow heat map with a readable text color."""
    mat = np.asarray(mat, dtype=float)
    lo, hi = np.nanmin(mat), np.nanmax(mat)
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            v = mat[i, j]
            ax.text(j, i, "–" if np.isnan(v) else fmt.format(v), ha="center", va="center", color="white" if (not np.isnan(v) and v > (lo + hi) / 2) else DARK)

### **1. Dataset and Preparation**

The same problem and the same **70/30 stratified split** of the other notebooks of the series: **malignant ( 1 ) × benign ( 0 )** tumours, the 10 *mean* attributes of the Breast Cancer dataset.

In [4]:
#Load the Breast Cancer Dataset ( Positive Class = Malignant )
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

data = load_breast_cancer(as_frame=True)
cols = [c for c in data.data.columns if c.startswith("mean ")]            # 10 "mean" attributes
X = data.data[cols]
y = (data.target == 0).astype(int)                                        # 1 = malignant, 0 = benign
CLASS_NAMES = ["benign", "malignant"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y)
print("train:", X_train.shape, "| test:", X_test.shape, "| malignant in train:", f"{y_train.mean():.1%}")
X_train.head(3)

train: (398, 10) | test: (171, 10) | malignant in train: 37.2%


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension
78,20.18,23.97,143.70,1245.0,0.129,0.345,0.375,0.160,0.291,0.081
330,16.03,15.51,105.80,793.2,0.095,0.137,0.120,0.070,0.178,0.060
378,13.66,15.15,88.27,580.6,0.083,0.075,0.042,0.025,0.179,0.059


In [ ]:
#Dictionary Used to Collect the Test Metrics of This Notebook ( the Cells Below Add Their Results to It )
results = {}

### **2. The Ingredients**

* **FOIL gain** of adding a condition to a rule that covers $p_0$ positives and $n_0$ negatives, and afterwards $p_1$ and $n_1$:

$$\text{gain} = p_1 \left( \log_2 \frac{p_1}{p_1 + n_1} - \log_2 \frac{p_0}{p_0 + n_0} \right)$$

* **Pruning value** of a rule on the pruning set: $v = \dfrac{p - n}{p + n}$ ( the final conditions are deleted if this value improves )
* **Description length ( DL )** of the rule set = cost of describing the **rules** + cost of describing the **exceptions** ( false positives among the covered cases and false negatives among the others ). Rule growth stops when the DL exceeds the smallest DL found by more than 64 bits

In [ ]:
#RIPPER ( IREP* with MDL Stopping ) Implementation for Numeric Attributes
from scipy.special import gammaln

def log2_binom(n, k):
    return (gammaln(n + 1) - gammaln(k + 1) - gammaln(n - k + 1)) / np.log(2)

class RIPPER:
    def __init__(self, prune_fraction=1 / 3, mdl_slack=64, prune=True, use_mdl=True, max_thresholds=40, seed=RANDOM_STATE):
        self.prune_fraction, self.mdl_slack, self.prune, self.use_mdl, self.max_thresholds, self.seed = prune_fraction, mdl_slack, prune, use_mdl, max_thresholds, seed

    def _mask(self, X, conds):
        m = np.ones(len(X), bool)
        for j, op, t in conds:
            m &= (X[:, j] <= t) if op == "<=" else (X[:, j] > t)
        return m

    def _candidates(self, X):
        out = []
        for j in range(X.shape[1]):
            values = np.unique(X[:, j])
            cuts = (values[:-1] + values[1:]) / 2
            if len(cuts) > self.max_thresholds:
                cuts = np.quantile(X[:, j], np.linspace(0.02, 0.98, self.max_thresholds))
            out += [(j, op, t) for t in cuts for op in ("<=", ">")]
        return out

    def _grow(self, X, y):
        conds, m = [], np.ones(len(y), bool)
        while True:
            p0, n0 = (m & y).sum(), (m & ~y).sum()
            if n0 == 0 or p0 == 0:
                break
            best, best_gain = None, 0.0
            for c in self.candidates_:
                m1 = m & self._mask(X, [c])
                p1, n1 = (m1 & y).sum(), (m1 & ~y).sum()
                if p1 == 0:
                    continue
                gain = p1 * (np.log2(p1 / (p1 + n1)) - np.log2(p0 / (p0 + n0)))
                if gain > best_gain:
                    best, best_gain = c, gain
            if best is None:
                break
            conds.append(best); m &= self._mask(X, [best])
        return conds

    def _prune(self, X, y, conds):
        best_k, best_v = len(conds), -np.inf
        for k in range(len(conds), 0, -1):
            m = self._mask(X, conds[:k])
            p, n = (m & y).sum(), (m & ~y).sum()
            v = (p - n) / (p + n) if p + n else -1.0
            if v >= best_v:                                                              # ties favour the shorter rule
                best_k, best_v = k, v
        return conds[:best_k]

    def _rule_dl(self, k):
        n = len(self.candidates_)
        return 0.5 * (log2_binom(n, k) + np.log2(k + 1))                                # which k of the n possible conditions ( subset encoding )

    def _total_dl(self, X, y, rules):
        covered = np.zeros(len(y), bool)
        for r in rules:
            covered |= self._mask(X, r)
        fp, fn = (covered & ~y).sum(), (~covered & y).sum()
        exceptions = np.log2(len(y) + 1) + log2_binom(covered.sum(), fp) + log2_binom((~covered).sum(), fn)
        return sum(self._rule_dl(len(r)) for r in rules) + exceptions

    def fit(self, X, y):
        X, y = np.asarray(X, dtype=float), np.asarray(y).astype(bool)
        rng = np.random.default_rng(self.seed)
        self.candidates_ = self._candidates(X)
        self.positive_ = 1
        rules, residual, min_dl = [], np.ones(len(y), bool), np.inf
        while (residual & y).sum() > 0:
            idx = np.flatnonzero(residual)
            pos_idx, neg_idx = idx[y[idx]], idx[~y[idx]]
            prune_set = np.concatenate([rng.choice(pos_idx, max(1, int(len(pos_idx) * self.prune_fraction)), replace=False) if len(pos_idx) > 1 else [],
                                        rng.choice(neg_idx, int(len(neg_idx) * self.prune_fraction), replace=False) if len(neg_idx) else []]).astype(int)
            grow_set = np.setdiff1d(idx, prune_set) if self.prune and len(prune_set) else idx
            conds = self._grow(X[grow_set], y[grow_set])
            if not conds:
                break
            if self.prune and len(prune_set):
                conds = self._prune(X[prune_set], y[prune_set], conds)
                m_p = self._mask(X[prune_set], conds)
                p, n = (m_p & y[prune_set]).sum(), (m_p & ~y[prune_set]).sum()
                if p + n == 0 or n / (p + n) > 0.5:                                      # error rate on the pruning set above 50%: stop
                    break
            new_rules = rules + [conds]
            dl = self._total_dl(X, y, new_rules)
            if self.use_mdl and dl > min_dl + self.mdl_slack:
                break
            min_dl = min(min_dl, dl)
            rules = new_rules
            residual &= ~self._mask(X, conds)
        if self.use_mdl:                                                                 # final clean-up: delete rules that increase the DL
            for i in range(len(rules) - 1, -1, -1):
                if len(rules) > 1 and self._total_dl(X, y, rules[:i] + rules[i + 1:]) < self._total_dl(X, y, rules):
                    rules = rules[:i] + rules[i + 1:]
        self.rules_ = rules
        covered = np.zeros(len(y), bool)
        self.stats_ = []
        for r in rules:                                                                  # ordered list: statistics on the cases not yet covered
            m = self._mask(X, r) & ~covered
            self.stats_.append((int((m & y).sum()), int((m & ~y).sum())))
            covered |= m
        self.default_p_ = ((y & ~covered).sum() + 1) / ((~covered).sum() + 2)
        return self

    def predict_proba(self, X):
        X = np.asarray(X, dtype=float)
        p1, free = np.full(len(X), self.default_p_), np.ones(len(X), bool)
        for r, (pos, neg) in zip(self.rules_, self.stats_):
            m = self._mask(X, r) & free
            p1[m] = (pos + 1) / (pos + neg + 2)
            free &= ~m
        return np.c_[1 - p1, p1]

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] > 0.5).astype(int)

    def n_rules(self):
        return len(self.rules_)

    def rules_table(self, names):
        return pd.DataFrame([{"IF": " AND ".join(f"{names[j]} {op} {t:.4g}" for j, op, t in r), "THEN": "malignant", "covers ( new )": pos + neg, "precision": pos / (pos + neg) if pos + neg else np.nan, "length": len(r)}
                             for r, (pos, neg) in zip(self.rules_, self.stats_)])

In [ ]:
#Fit RIPPER on the Training Set and Show the Rule List ( Default Rule: benign )
ripper = RIPPER().fit(X_train, y_train)
proba_ripper = ripper.predict_proba(X_test)[:, 1]
pred_ripper = ripper.predict(X_test)
print(f"{ripper.n_rules()} rules for the malignant class + default rule ( benign )")
results["RIPPER"] = report_binary(y_test, pred_ripper, proba_ripper, "RIPPER", CLASS_NAMES)
display(results["RIPPER"].to_frame().T.round(3))
pd.set_option("display.max_colwidth", 110)
ripper.rules_table(list(X_train.columns)).round(3)

### **3. Quality Metrics Specific to RIPPER**

In [ ]:
#Effect of Pruning and of the MDL Stopping Rule ( Same Data, Same Seed )
variants = {"IREP* with pruning and MDL ( default )": dict(), "no pruning": dict(prune=False), "no MDL stopping": dict(use_mdl=False), "no pruning, no MDL": dict(prune=False, use_mdl=False)}
rows = []
for label, params in variants.items():
    m = RIPPER(**params).fit(X_train, y_train)
    rows.append({"variant": label, "rules": m.n_rules(), "mean length": np.mean([len(r) for r in m.rules_]), "test accuracy": accuracy_score(y_test, m.predict(X_test)), "test ROC AUC": roc_auc_score(y_test, m.predict_proba(X_test)[:, 1])})
pd.DataFrame(rows).round(3)

In [ ]:
#Stability: 20 Random Splits Between the Growing Set and the Pruning Set ( Different Seeds )
rows = []
for seed in range(20):
    m = RIPPER(seed=seed).fit(X_train, y_train)
    used = {(X_train.columns[j]) for r in m.rules_ for j, _, _ in r}
    first = X_train.columns[m.rules_[0][0][0]] if m.rules_ else None
    rows.append({"seed": seed, "rules": m.n_rules(), "test accuracy": accuracy_score(y_test, m.predict(X_test)), "test ROC AUC": roc_auc_score(y_test, m.predict_proba(X_test)[:, 1]), "attributes used": len(used), "first condition on": first})
stab = pd.DataFrame(rows)
print(f"rules: {stab['rules'].mean():.1f} ± {stab['rules'].std():.1f} ( min {stab['rules'].min()}, max {stab['rules'].max()} ) | test accuracy: {stab['test accuracy'].mean():.3f} ± {stab['test accuracy'].std():.3f} | test ROC AUC: {stab['test ROC AUC'].mean():.3f} ± {stab['test ROC AUC'].std():.3f}")
print("attribute of the first condition of the first rule:", stab["first condition on"].value_counts().to_dict())
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
axes[0].hist(stab["rules"], bins=np.arange(stab["rules"].min() - 0.5, stab["rules"].max() + 1.5), color=DARK, rwidth=0.8); axes[0].set_xlabel("rules"); axes[0].set_ylabel("runs"); axes[0].set_title("Number of rules over 20 seeds")
axes[1].hist(stab["test accuracy"], bins=10, color=RED, rwidth=0.8); axes[1].set_xlabel("test accuracy"); axes[1].set_title("Test accuracy over 20 seeds")
plt.tight_layout(); plt.show()

In [ ]:
#Soundness Check: the Statistics of the Rule List Recomputed Independently on the Training Set ( Ordered First-Match Semantics )
covered = np.zeros(len(y_train), bool)
mismatch = 0
Xtr = X_train.to_numpy(dtype=float)
for r, (pos, neg) in zip(ripper.rules_, ripper.stats_):
    m = np.ones(len(Xtr), bool)
    for j, op, t in r:
        m &= (Xtr[:, j] <= t) if op == "<=" else (Xtr[:, j] > t)
    m &= ~covered
    mismatch += int(pos != (m & (y_train.to_numpy() == 1)).sum()) + int(neg != (m & (y_train.to_numpy() == 0)).sum())
    covered |= m
print(f"rules checked: {len(ripper.rules_)} | counts that disagree with the independent recount: {mismatch}")

### **4. Evaluation**

In [ ]:
#Reference Models on the Same Split ( Is the Technique Better than a Trivial or a Standard Model? )
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

references = {"Majority class": DummyClassifier(strategy="prior"),
              "Decision tree ( depth 3 )": DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE),
              "Logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))}
reference_rows = {}
for ref_name, model in references.items():
    model.fit(X_train, y_train)
    reference_rows[ref_name] = report_binary(y_test, model.predict(X_test), model.predict_proba(X_test)[:, 1], ref_name, CLASS_NAMES, plot=False)
metric_cols = ["accuracy", "precision", "recall", "F1", "ROC AUC", "PR AUC"]
pd.concat([pd.DataFrame(results).T[metric_cols], pd.DataFrame(reference_rows).T[metric_cols]]).round(3)

In [ ]:
#Repeated Stratified CV on the Training Set ( 5 Folds x 3 ): RIPPER and the Reference Models
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate
cv_rep = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=RANDOM_STATE)
accs, aucs, sizes = [], [], []
for tr, te in cv_rep.split(X_train, y_train):
    m = RIPPER().fit(X_train.iloc[tr], y_train.iloc[tr])
    p = m.predict_proba(X_train.iloc[te])[:, 1]
    accs.append(((p > 0.5).astype(int) == y_train.iloc[te].to_numpy()).mean()); aucs.append(roc_auc_score(y_train.iloc[te], p)); sizes.append(m.n_rules())
cv_rows = {"RIPPER": {"accuracy ( mean )": np.mean(accs), "accuracy ( std )": np.std(accs), "ROC AUC ( mean )": np.mean(aucs), "ROC AUC ( std )": np.std(aucs)}}
for ref_name, model in references.items():
    res = cross_validate(model, X_train, y_train, cv=cv_rep, scoring=["accuracy", "roc_auc"])
    cv_rows[ref_name] = {"accuracy ( mean )": res["test_accuracy"].mean(), "accuracy ( std )": res["test_accuracy"].std(), "ROC AUC ( mean )": res["test_roc_auc"].mean(), "ROC AUC ( std )": res["test_roc_auc"].std()}
print(f"RIPPER rules per fold: {np.mean(sizes):.1f} ± {np.std(sizes):.1f}")
pd.DataFrame(cv_rows).T.round(3)

In [ ]:
#Uncertainty of the Test Metrics ( Bootstrap of the Test Cases, 1,000 Resamples )
def bootstrap_ci(y_true, pred, score, n_boot=1000, seed=RANDOM_STATE):
    rng_b = np.random.default_rng(seed)
    y_true, pred, score = np.asarray(y_true), np.asarray(pred), np.asarray(score)
    acc, auc = [], []
    for _ in range(n_boot):
        i = rng_b.integers(0, len(y_true), len(y_true))
        if len(np.unique(y_true[i])) < 2:
            continue
        acc.append((pred[i] == y_true[i]).mean())
        auc.append(roc_auc_score(y_true[i], score[i]))
    return pd.DataFrame({"accuracy": np.percentile(acc, [2.5, 50, 97.5]), "ROC AUC": np.percentile(auc, [2.5, 50, 97.5])}, index=["2.5%", "median", "97.5%"]).T

print("the test set has", len(y_test), "cases: differences smaller than these intervals are not reliable")
bootstrap_ci(y_test, pred_ripper, proba_ripper).round(3)

In [ ]:
#Calibration of the Predicted Probabilities ( Brier Score and Reliability Curve )
from sklearn.metrics import brier_score_loss
from sklearn.calibration import calibration_curve

proba_ripper_ARRAY = np.asarray(proba_ripper, dtype=float)
frac_pos, mean_pred = calibration_curve(y_test, proba_ripper_ARRAY, n_bins=5, strategy="quantile")                       # bins with the same number of cases
print(f"Brier score: {brier_score_loss(y_test, proba_ripper_ARRAY):.4f} | Brier score of a model that always predicts the training prevalence: {brier_score_loss(y_test, np.full(len(y_test), y_train.mean())):.4f} ( lower is better )")
fig, ax = plt.subplots(figsize=(4.6, 4.2))
ax.plot([0, 1], [0, 1], color=GRAY, ls="--", label="perfect calibration"); ax.plot(mean_pred, frac_pos, color=RED, marker="o", label="RIPPER")
ax.set_xlabel("mean predicted probability ( 5 bins of equal size )"); ax.set_ylabel("observed share of malignant"); ax.set_title("Reliability curve"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

### **5. Takeaways**

* RIPPER works on the **numeric attributes**: the thresholds are learned, so there is no discretisation step and no information is lost in bins
* **Pruning is the main mechanism that keeps the list short** ( 5 rules without it against 3 with it ); the **MDL stopping rule** matters mostly when pruning is off ( 9 rules against 5 ), see the variants of section 3. In both cases the description length, not a user-chosen number, decides when to stop
* The list depends on the **random split between growing and pruning sets** ( stability table ): the number of rules varies between runs ( 2 to 4 ), while the first condition is almost always on *mean concave points*; read the rules as one explanation among similar ones
* This is the **IREP\*** core: the full RIPPER also **optimises** each rule ( replacement and revision ) in 2 extra passes, which usually shortens the list and improves it a little; that step is not implemented here
* As in the other notebooks, the cross-validation was done on the training set only and the test set is small: look at the bootstrap intervals before ranking techniques